In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as UTC datetimes
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

log_df = pd.read_csv("log.csv")
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

error_df = pd.read_csv("error_logs.csv")
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Helper to aggregate: for a given df, group_cols, name_col produce summary
def aggregate_timeseries(df, group_cols, value_col, name_for_value='value'):
    agg = df.groupby(group_cols).agg(
        count=(value_col, "count"),
        min_timestamp=("timestamp", "min"),
        max_timestamp=("timestamp", "max"),
        p50=(value_col, lambda x: x.quantile(0.5)),
        p95=(value_col, lambda x: x.quantile(0.95))
    ).reset_index()
    # sort by count desc and return top 20
    agg = agg.sort_values("count", ascending=False).head(20)
    # round numeric percentiles for compactness
    agg['p50'] = agg['p50'].astype(float).round(6)
    agg['p95'] = agg['p95'].astype(float).round(6)
    return agg

# 1) metric summary
metric_summary = aggregate_timeseries(metric_df, ["cmdb_id", "kpi_name"], "value")

# 2) trace summary
trace_summary = aggregate_timeseries(trace_df, ["cmdb_id", "trace_name"], "value")

# 3) log summary
log_summary = aggregate_timeseries(log_df, ["cmdb_id", "log_name"], "value")

# 4) error_logs summary: total count, top cmdb_id counts (top20), and up to 5 sample rows
error_total = len(error_df)
cmdb_counts_df = error_df['cmdb_id'].value_counts().head(20).reset_index()
cmdb_counts_df.columns = ['cmdb_id', 'count']
error_sample = error_df[['timestamp', 'cmdb_id', 'message']].head(5)

# Display the four compact summaries (variables returned)
metric_summary, trace_summary, log_summary, error_total, cmdb_counts_df, error_sample

```
Out[1]:
```
Summary of the telemetry (UTC 2024-01-24 14:25–14:49):

- Data coverage: All metric/trace/log series shown have 25 samples (minute-level) covering 2024-01-24T14:25:00Z → 2024-01-24T14:49:00Z. There are no error log entries (error_logs count = 0).
- Logs: High request volumes observed for ts-basic-service (log.row_count P50 ≈ 2,414) and notable row counts for ts-travel2-service and ts-route-service. log.error_count is zero for the reported services.
- Metrics: Many services show stable metrics, but ts-travel2-service stands out with a much higher CPU metric (P50 ≈ 4.84, P95 ≈ 5.09) compared with other services that have CPU values << 1.
- Traces: ts-travel2-service trace activity is high (several trace row_count metrics: P50s in the thousands) and some trace paths to/from ts-travel2-service have larger durations (e.g., from ts-admin-travel-service → ts-travel2-service: duration P50 ≈ 0.122s, P95 ≈ 0.201s), higher than most other inter-service traces.
- Interpretation / likely root cause: There is no evidence of application errors in logs, but ts-travel2-service shows both elevated CPU and elevated trace durations and very high request/row counts — consistent with a performance hotspot caused by high CPU/load on ts-travel2-service.
- Recommended next steps: focus investigation on ts-travel2-service (thread/CPU profiling, slow-path traces, database or downstream calls), consider throttling or scaling, and inspect detailed traces for the slow interactions (especially calls from admin-travel-service).

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count             min_timestamp             max_timestamp           p50           p95
368           ts-voucher-service      socket     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  4.000000e+00  4.000000e+00
0    ts-admin-basic-info-service         cpu     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  4.317150e-01  1.587257e+00
1    ts-admin-basic-info-service      diskio     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  5.069296e+04  6.049452e+04
2    ts-admin-basic-info-service  latency-50     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  1.689600e-02  1.739500e-02
3    ts-admin-basic-info-service  latency-90     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  2.338700e-02  2.519700e-02
4    ts-admin-basic-info-service         mem     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  2.367493e+08  2.373338e+08
5    ts-admin-basic-info-service      socket     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  7.533333e+00  8.000000e+00
6    ts-admin-basic-info-service    workload     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  6.432330e-01  7.297630e-01
7         ts-admin-order-service         cpu     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  9.507600e-02  1.066100e-01
8         ts-admin-order-service      diskio     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  3.996008e+04  4.805086e+04
9         ts-admin-order-service         mem     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  1.934050e+08  1.934070e+08
10        ts-admin-order-service      socket     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  3.100000e+00  3.443333e+00
11        ts-admin-route-service         cpu     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  9.282400e-02  1.022460e-01
12        ts-admin-route-service      diskio     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  3.943124e+04  4.919884e+04
13        ts-admin-route-service         mem     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  1.960068e+08  1.960082e+08
14        ts-admin-route-service      socket     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  3.000000e+00  3.357143e+00
15       ts-admin-travel-service         cpu     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  2.969420e-01  5.327780e-01
329             ts-travel2-mongo      socket     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  5.000000e+00  5.000000e+00
330           ts-travel2-service         cpu     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  4.837577e+00  5.089090e+00
331           ts-travel2-service      diskio     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  5.047070e+04  6.024385e+04,                 cmdb_id                                        trace_name  count             min_timestamp             max_timestamp          p50          p95
728  ts-travel2-service     trace.to_ts-order-other-service.duration_mean     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.006765     0.007616
729  ts-travel2-service      trace.to_ts-order-other-service.duration_p95     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.012369     0.014756
730  ts-travel2-service        trace.to_ts-order-other-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.000000     0.000000
731  ts-travel2-service         trace.to_ts-order-other-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    77.000000    82.000000
732  ts-travel2-service           trace.to_ts-route-service.duration_mean     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.003401     0.004816
733  ts-travel2-service            trace.to_ts-route-service.duration_p95     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.004135     0.004977
734  ts-travel2-service              trace.to_ts-route-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.000000     0.000000
735  ts-travel2-service               trace.to_ts-route-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00   584.000000   613.800000
720  ts-travel2-service          trace.from_ts-seat-service.duration_mean     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.014317     0.020297
721  ts-travel2-service           trace.from_ts-seat-service.duration_p95     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.038900     0.065268
722  ts-travel2-service             trace.from_ts-seat-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.000000     0.000000
723  ts-travel2-service              trace.from_ts-seat-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00   311.000000   333.600000
724  ts-travel2-service       trace.from_ts-travel2-service.duration_mean     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.011283     0.013756
725  ts-travel2-service        trace.from_ts-travel2-service.duration_p95     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.050091     0.068390
726  ts-travel2-service          trace.from_ts-travel2-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.000000     0.000000
727  ts-travel2-service           trace.from_ts-travel2-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  2541.000000  2703.600000
712  ts-travel2-service  trace.from_ts-admin-travel-service.duration_mean     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.122405     0.201745
713  ts-travel2-service   trace.from_ts-admin-travel-service.duration_p95     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.201123     0.265798
714  ts-travel2-service     trace.from_ts-admin-travel-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.000000     0.000000
715  ts-travel2-service      trace.from_ts-admin-travel-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    10.000000    11.000000,                         cmdb_id         log_name  count             min_timestamp             max_timestamp     p50     p95
0   ts-admin-basic-info-service  log.error_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.0     0.0
1   ts-admin-basic-info-service    log.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    29.0    32.6
2       ts-admin-travel-service  log.error_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.0     0.0
3       ts-admin-travel-service    log.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    30.0    33.0
7          ts-assurance-service    log.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     9.0    16.6
6          ts-assurance-service  log.error_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.0     0.0
10              ts-auth-service  log.error_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.0     0.0
11              ts-auth-service    log.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00   334.0   349.6
23           ts-consign-service    log.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    49.0    75.6
22           ts-consign-service  log.error_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.0     0.0
13             ts-basic-service    log.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  2414.0  2622.4
12             ts-basic-service  log.error_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.0     0.0
27          ts-contacts-service    log.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    36.0    60.8
26          ts-contacts-service  log.error_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.0     0.0
62             ts-price-service  log.error_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.0     0.0
63             ts-price-service    log.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00   264.0   288.0
67             ts-route-service    log.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  1005.0  1094.4
66             ts-route-service  log.error_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.0     0.0
49             ts-order-service    log.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00   369.0   455.6
48             ts-order-service  log.error_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00     0.0     0.0, 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

# Reuse loaded DataFrames: metric_df, trace_df, log_df (timestamps already parsed)
import pandas as pd

# 1) Per-file counts per cmdb_id
metric_counts = metric_df['cmdb_id'].value_counts().reset_index()
metric_counts.columns = ['cmdb_id', 'row_count']
metric_counts.insert(0, 'source_file', 'metric.csv')
metric_counts = metric_counts.sort_values('row_count', ascending=False).head(50)

trace_counts = trace_df['cmdb_id'].value_counts().reset_index()
trace_counts.columns = ['cmdb_id', 'row_count']
trace_counts.insert(0, 'source_file', 'trace.csv')
trace_counts = trace_counts.sort_values('row_count', ascending=False).head(50)

log_counts = log_df['cmdb_id'].value_counts().reset_index()
log_counts.columns = ['cmdb_id', 'row_count']
log_counts.insert(0, 'source_file', 'log.csv')
log_counts = log_counts.sort_values('row_count', ascending=False).head(50)

# 2) Combined unique sorted list of cmdb_id values observed across all three files
combined_unique = sorted(set(metric_df['cmdb_id'].unique()) | set(trace_df['cmdb_id'].unique()) | set(log_df['cmdb_id'].unique()))
combined_cmdbs = pd.DataFrame({'cmdb_id': combined_unique}).head(50)

# Return the summaries (as variables for IPython display)
metric_counts, trace_counts, log_counts, combined_cmdbs

```
Out[2]:
```
Summary of results (plain English):

- metric.csv (per-cmdb row counts): Top entries (from the returned top-50) show many services with 200 rows each — e.g., ts-admin-travel-service, ts-order-other-service, ts-contacts-service, ts-basic-service, ts-ticketinfo-service. The returned metric count table included 50 cmdb_id rows (top 50 by row_count).

- trace.csv (per-cmdb row counts): The largest trace sources are:
  - root: 1,384 rows
  - ts-preserve-service: 1,232 rows
  - ts-travel-service: 1,196 rows
  - ts-preserve-other-service: 1,108 rows
  - ts-travel2-service: 1,088 rows
  The trace summary returned 28 cmdb_id rows.

- log.csv (per-cmdb row counts): Many services have 50 log rows in the returned table. Examples with row_count = 50 include ts-admin-basic-info-service, ts-admin-travel-service, ts-assurance-service, ts-auth-service, ts-consign-service. The returned log summary contained 46 cmdb_id rows.

- Combined unique cmdb_id list (sorted): The combined set of cmdb_id values observed across metric.csv, trace.csv, and log.csv includes (sample / first 50 shown): root, ts-admin-basic-info-service, ts-admin-order-service, ts-admin-route-service, ts-admin-travel-service, ... , ts-route-mongo, ts-route-plan-service, ts-route-service, ts-seat-service, ts-security-mongo. (The returned combined list was limited to the first 50 sorted cmdb_ids.)

If you want any of these lists expanded or the full per-file count tables printed (or the exact total number of unique cmdb_ids), tell me which one and I will display it.

The original code execution output of IPython Kernel is also provided below for reference:

(   source_file                  cmdb_id  row_count
0   metric.csv  ts-admin-travel-service        200
1   metric.csv   ts-order-other-service        200
2   metric.csv      ts-contacts-service        200
3   metric.csv         ts-basic-service        200
4   metric.csv    ts-ticketinfo-service        200
..         ...                      ...        ...
45  metric.csv            ts-food-mongo        100
46  metric.csv        ts-food-map-mongo        100
47  metric.csv            ts-auth-mongo        100
48  metric.csv   ts-consign-price-mongo        100
49  metric.csv         ts-consign-mongo        100

[50 rows x 3 columns],    source_file                    cmdb_id  row_count
0    trace.csv                       root       1384
1    trace.csv        ts-preserve-service       1232
2    trace.csv          ts-travel-service       1196
3    trace.csv  ts-preserve-other-service       1108
4    trace.csv         ts-travel2-service       1088
..         ...                        ...        ...
23   trace.csv            ts-user-service        384
24   trace.csv        ts-food-map-service        300
25   trace.csv   ts-consign-price-service        300
26   trace.csv            ts-auth-service        300
27   trace.csv         ts-payment-service        300

[28 rows x 3 columns],    source_file                      cmdb_id  row_count
0      log.csv  ts-admin-basic-info-service         50
1      log.csv      ts-admin-travel-service         50
2      log.csv         ts-assurance-service         50
3      log.csv              ts-auth-service         50
4      log.csv           ts-consign-service         50
..         ...                          ...        ...
41     log.csv               ts-train-mongo         48
42     log.csv                ts-user-mongo         48
43     log.csv              ts-travel-mongo         48
44     log.csv    ts-preserve-other-service         44
45     log.csv            ts-config-service          2

[46 rows x 3 columns],                         cmdb_id
0                          root
1   ts-admin-basic-info-service
2        ts-admin-order-service
3        ts-admin-route-service
4       ts-admin-travel-service
..                          ...
45               ts-route-mongo
46        ts-route-plan-service
47             ts-route-service
48              ts-seat-service
49            ts-security-mongo

[50 rows x 1 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Reuse metric_df loaded previously. Analyze candidate components with global thresholds then windowed anomalies.

import pandas as pd
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Ensure metric_df has timestamp parsed (should be from prior steps). If not, parse:
if metric_df['timestamp'].dtype == 'int64' or metric_df['timestamp'].dtype == 'int32':
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# 1) Global thresholds (use full series) for candidate components
metric_candidates = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

global_agg = metric_candidates.groupby(['cmdb_id', 'kpi_name']).agg(
    count=('value', 'count'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
    p50=('value', lambda x: float(x.quantile(0.5))),
    p95=('value', lambda x: float(x.quantile(0.95)))
).reset_index()

# Round percentiles for compactness
global_agg['p50'] = global_agg['p50'].round(6)
global_agg['p95'] = global_agg['p95'].round(6)

# 2) Filter to incident window and compute windowed stats and anomalies
window_start = pd.Timestamp('2024-01-24 14:22:25', tz='UTC')
window_end   = pd.Timestamp('2024-01-24 14:52:25', tz='UTC')

window_df = metric_candidates[(metric_candidates['timestamp'] >= window_start) & 
                              (metric_candidates['timestamp'] <= window_end)].copy()

# Attach global p95 to window rows for anomaly detection
window_df = window_df.merge(global_agg[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')

# mark anomalies where value >= global p95
window_df['is_anomaly'] = window_df['value'] >= window_df['p95']

# window-level aggregations
window_stats_count_max = window_df.groupby(['cmdb_id','kpi_name']).agg(
    window_count=('value','count'),
    window_max=('value','max')
).reset_index()

window_anom_counts = window_df.groupby(['cmdb_id','kpi_name']).agg(
    window_anomaly_count=('is_anomaly','sum')
).reset_index()

# earliest anomaly timestamp per group
anom_rows = window_df[window_df['is_anomaly']]
earliest_anom = anom_rows.groupby(['cmdb_id','kpi_name']).agg(
    earliest_anomaly_timestamp=('timestamp','min')
).reset_index()

# Merge window stats together
window_stats = window_stats_count_max.merge(window_anom_counts, on=['cmdb_id','kpi_name'], how='left')
window_stats = window_stats.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# For groups with no window rows, ensure they appear with zeros/nulls by merging with global_agg keys
result = global_agg.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with 0 where appropriate
result['window_count'] = result['window_count'].fillna(0).astype(int)
result['window_anomaly_count'] = result['window_anomaly_count'].fillna(0).astype(int)
# window_max may remain NaN if no data; earliest_anomaly_timestamp remains NaT if none

# Reorder and select desired columns
result = result[['cmdb_id','kpi_name','count','min_timestamp','max_timestamp','p50','p95',
                 'window_count','window_anomaly_count','window_max','earliest_anomaly_timestamp']]

# Sort by window_anomaly_count desc then cmdb_id, limit to top 50 rows
result_df = result.sort_values(['window_anomaly_count','cmdb_id'], ascending=[False, True]).head(50)

# Display the compact result (top 50)
result_df

```
Out[3]:
```
Summary of the analysis (global thresholds computed on full series; window = 2024-01-24 14:22:25 → 14:52:25 UTC):

What was done
- For each (cmdb_id, kpi_name) I computed global P50 and P95 from the full metric series, then counted window points that exceed the global P95 (anomalies) inside the incident window.

Key findings (compact)
- ts-train-service — metric "error":
  - global p95 = 0, full count = 25; window_count = 25, window_anomaly_count = 25, earliest = 2024-01-24T14:25:00Z.
  - Note: p95 = 0 means all zero values meet >= p95, so these are not meaningful “spikes” — treat as artifact of zero-valued KPI.

- ts-travel-service — metric "error":
  - same situation as above: p95 = 0, window_count = 25, window_anomaly_count = 25, earliest = 2024-01-24T14:25:00Z (artifact).

- ts-route-service — metric "error":
  - p95 = 0, window_count = 8, window_anomaly_count = 8, earliest = 2024-01-24T14:42:00Z (artifact).

- ts-auth-service — cpu:
  - global p95 ≈ 15.5729, full count = 25
  - window_count = 25, window_anomaly_count = 2
  - window_max ≈ 16.0705, earliest anomaly 2024-01-24T14:28:00Z
  - Interpretation: CPU exceeded its historical P95 twice in the incident window — actionable.

- ts-auth-service — diskio:
  - global p95 ≈ 58643, full count = 25
  - window_count = 25, window_anomaly_count = 2
  - window_max ≈ 59747, earliest anomaly 2024-01-24T14:27:00Z
  - Interpretation: two disk I/O points exceed global P95.

- ts-travel-service — multiple KPIs (examples):
  - Several KPIs (latency-50, latency-90, mem, socket, workload, etc.) show window_anomaly_count = 2 each.
  - Examples: latency-90 window_max ≈ 0.312s (earliest 14:38), mem window_max ≈ 2.7356e8 (earliest 14:28), socket window_max ≈ 18.55 (earliest 14:45), workload window_max ≈ 12.30 (earliest 14:35).
  - Interpretation: multiple resource/latency KPIs in ts-travel-service exceeded their global P95 twice during the window — consistent with transient load/performance issues.

Important caveat
- Many "error" KPIs had global P95 = 0, which causes the >= P95 rule to mark every zero as an “anomaly.” Those should not be interpreted as error spikes; instead treat zero-valued error KPIs separately (e.g., look for >0 error counts rather than >= p95 when p95 == 0).

Actionable recommendations
- Investigate ts-auth-service CPU and disk I/O around 14:27–14:28 UTC (two exceedances).
- Investigate ts-travel-service for elevated latency, memory, socket and workload around 14:28–14:45 UTC (multiple KPIs had two P95 exceedances).
- Ignore or re-evaluate "anomalies" for error KPIs with p95 == 0 (they are artifacts of the threshold rule); instead look for non-zero error counts/events.
- If confirming impact, collect detailed traces/profiles for ts-auth-service and ts-travel-service in the noted time ranges and consider scaling or throttling if load-related.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count             min_timestamp             max_timestamp           p50           p95  window_count  window_anomaly_count    window_max earliest_anomaly_timestamp
24   ts-train-service       error     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.000000e+00  0.000000e+00            25                    25  0.000000e+00  2024-01-24 14:25:00+00:00
32  ts-travel-service       error     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.000000e+00  0.000000e+00            25                    25  0.000000e+00  2024-01-24 14:25:00+00:00
16   ts-route-service       error      8 2024-01-24 14:42:00+00:00 2024-01-24 14:49:00+00:00  0.000000e+00  0.000000e+00             8                     8  0.000000e+00  2024-01-24 14:42:00+00:00
0     ts-auth-service         cpu     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  1.400360e+01  1.572919e+01            25                     2  1.607050e+01  2024-01-24 14:28:00+00:00
1     ts-auth-service      diskio     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  5.122154e+04  5.864329e+04            25                     2  5.974769e+04  2024-01-24 14:27:00+00:00
..                ...         ...    ...                       ...                       ...           ...           ...           ...                   ...           ...                        ...
33  ts-travel-service  latency-50     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  1.765800e-02  2.084900e-02            25                     2  2.107396e-02  2024-01-24 14:43:00+00:00
34  ts-travel-service  latency-90     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  2.320010e-01  2.762620e-01            25                     2  3.123616e-01  2024-01-24 14:38:00+00:00
35  ts-travel-service         mem     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  2.729442e+08  2.734624e+08            25                     2  2.735636e+08  2024-01-24 14:28:00+00:00
36  ts-travel-service      socket     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  1.635000e+01  1.814286e+01            25                     2  1.855000e+01  2024-01-24 14:45:00+00:00
37  ts-travel-service    workload     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  9.198967e+00  1.030383e+01            25                     2  1.203007e+01  2024-01-24 14:35:00+00:00

[38 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Candidate components
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Ensure trace_df timestamps are parsed as UTC datetimes (reuse existing variable if present)
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
else:
    # if timestamp not yet converted to datetime, convert it
    if not pd.api.types.is_datetime64_any_dtype(trace_df['timestamp']):
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Filter to candidate components for analysis
trace_candidates = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()

# 1) Compute global (full-series) aggregations and thresholds per (cmdb_id, trace_name)
global_trace_agg = trace_candidates.groupby(['cmdb_id', 'trace_name']).agg(
    count=('value', 'count'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
    p50=('value', lambda x: float(x.quantile(0.5))),
    p95=('value', lambda x: float(x.quantile(0.95)))
).reset_index()

global_trace_agg['p50'] = global_trace_agg['p50'].round(6)
global_trace_agg['p95'] = global_trace_agg['p95'].round(6)

# 2) Filter to incident window and compute windowed stats using global p95 for anomaly detection
window_start = pd.Timestamp('2024-01-24 14:22:25', tz='UTC')
window_end   = pd.Timestamp('2024-01-24 14:52:25', tz='UTC')

window_trace = trace_candidates[(trace_candidates['timestamp'] >= window_start) &
                                (trace_candidates['timestamp'] <= window_end)].copy()

# Attach global p95 to window rows
window_trace = window_trace.merge(global_trace_agg[['cmdb_id','trace_name','p95']],
                                  on=['cmdb_id','trace_name'], how='left')

# Mark anomalies where value >= global p95
window_trace['is_anomaly'] = window_trace['value'] >= window_trace['p95']

# Window-level aggregations
win_counts = window_trace.groupby(['cmdb_id','trace_name']).agg(
    window_count=('value','count'),
    window_max=('value','max')
).reset_index()

win_anom = window_trace.groupby(['cmdb_id','trace_name']).agg(
    window_anomaly_count=('is_anomaly','sum')
).reset_index()

earliest_anom = window_trace[window_trace['is_anomaly']].groupby(['cmdb_id','trace_name']).agg(
    earliest_anomaly_timestamp=('timestamp','min')
).reset_index()

# Merge window stats with global aggregations
trace_result = global_trace_agg.merge(win_counts, on=['cmdb_id','trace_name'], how='left') \
                               .merge(win_anom, on=['cmdb_id','trace_name'], how='left') \
                               .merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for window counts with zeros and ensure types
trace_result['window_count'] = trace_result['window_count'].fillna(0).astype(int)
trace_result['window_anomaly_count'] = trace_result['window_anomaly_count'].fillna(0).astype(int)
# window_max may stay NaN if no data in window; earliest_anomaly_timestamp remains NaT if none

# Select and order columns as requested
trace_result = trace_result[['cmdb_id','trace_name','count','min_timestamp','max_timestamp','p50','p95',
                             'window_count','window_anomaly_count','window_max','earliest_anomaly_timestamp']]

# 3) Sort by window_anomaly_count desc then cmdb_id and limit to top 20 rows
trace_result_df = trace_result.sort_values(['window_anomaly_count','cmdb_id'], ascending=[False, True]).head(20)

# Return the compact result
trace_result_df

```
Out[4]:
```
Summary of trace analysis (candidates: ts-auth, ts-order, ts-route, ts-train, ts-travel; window = 2024-01-24 14:22:25 → 14:52:25 UTC):

- What the results show:
  - For the candidate services, the trace series returned here are primarily various "error_rate" trace metrics.
  - Global P50 and P95 for these error_rate series are 0.0 (computed on the full series).
  - Because P95 == 0, every in-window value (which is also 0.0) satisfies value >= P95, so window_anomaly_count = window_count (often 25) and earliest_anomaly_timestamp = 2024-01-24T14:25:00Z.
  - window_max for these series is 0.0 — there are no positive error rates in the window.

- Interpretation / caveat:
  - The apparent "anomalies" are an artifact of using >= global P95 when the P95 is zero. That does not indicate real error spikes.
  - In short: there are no non-zero error-rate spikes detected for these components in the incident window.

- Recommended next steps:
  - Re-evaluate anomaly criteria for error_rate series: consider flagging only values > 0 (or require p95 > 0) rather than >= p95 when p95 == 0.
  - Inspect other trace metrics (duration_mean, duration_p95, row_count) for latency/throughput anomalies — these are more informative for performance issues than zero-valued error_rate traces.
  - If you want, I can rerun the same windowed check but (a) using a >0 rule for error_rate, or (b) analyzing duration and row_count traces for these services.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                       trace_name  count             min_timestamp             max_timestamp  p50  p95  window_count  window_anomaly_count  window_max earliest_anomaly_timestamp
2    ts-auth-service                       trace.from_root.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
6    ts-auth-service            trace.from_ts-auth-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
10   ts-auth-service              trace.to_ts-auth-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
14  ts-order-service                       trace.from_root.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
18  ts-order-service  trace.from_ts-inside-payment-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
22  ts-order-service           trace.from_ts-order-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
30  ts-order-service            trace.from_ts-seat-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
34  ts-order-service        trace.from_ts-security-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
38  ts-order-service          trace.from_ts-travel-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
42  ts-order-service             trace.to_ts-order-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
46  ts-order-service           trace.to_ts-station-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
50  ts-route-service           trace.from_ts-basic-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
54  ts-route-service           trace.from_ts-route-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
58  ts-route-service          trace.from_ts-travel-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
62  ts-route-service         trace.from_ts-travel2-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
66  ts-route-service             trace.to_ts-route-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
70  ts-train-service           trace.from_ts-basic-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
74  ts-train-service           trace.from_ts-train-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
78  ts-train-service          trace.from_ts-travel-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00
82  ts-train-service         trace.from_ts-travel2-service.error_rate     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  0.0  0.0            25                    25         0.0  2024-01-24 14:25:00+00:00```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse trace_df if present; ensure timestamp is parsed as UTC datetime
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
if not pd.api.types.is_datetime64_any_dtype(trace_df['timestamp']):
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure value is numeric
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Candidate components
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Filter to candidate components
trace_candidates = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()

# 1) Select trace_name containing 'duration' or 'row_count' (case-insensitive)
mask = trace_candidates['trace_name'].str.contains('duration', case=False, na=False) | \
       trace_candidates['trace_name'].str.contains('row_count', case=False, na=False)
trace_sel = trace_candidates[mask].copy()

# 2) Global (full-series) aggregations per (cmdb_id, trace_name) -> count, min/max ts, p50, p95
global_trace_agg = trace_sel.groupby(['cmdb_id','trace_name']).agg(
    count=('value','count'),
    min_timestamp=('timestamp','min'),
    max_timestamp=('timestamp','max'),
    p50=('value', lambda x: float(x.quantile(0.5)) if len(x.dropna())>0 else float('nan')),
    p95=('value', lambda x: float(x.quantile(0.95)) if len(x.dropna())>0 else float('nan'))
).reset_index()

global_trace_agg['p50'] = global_trace_agg['p50'].round(6)
global_trace_agg['p95'] = global_trace_agg['p95'].round(6)

# 3) Filter to incident window and compute windowed stats using global p95 for anomaly detection
window_start = pd.Timestamp('2024-01-24 14:22:25', tz='UTC')
window_end   = pd.Timestamp('2024-01-24 14:52:25', tz='UTC')

window_trace = trace_sel[(trace_sel['timestamp'] >= window_start) & (trace_sel['timestamp'] <= window_end)].copy()

# Attach global p95 to window rows
window_trace = window_trace.merge(global_trace_agg[['cmdb_id','trace_name','p95']], on=['cmdb_id','trace_name'], how='left')

# Mark anomalies where value >= global p95 (respecting rule to compute p95 on full series first)
window_trace['is_anomaly'] = window_trace['value'] >= window_trace['p95']

# Window-level aggregations
win_counts = window_trace.groupby(['cmdb_id','trace_name']).agg(
    window_count=('value','count'),
    window_max=('value','max')
).reset_index()

win_anom = window_trace.groupby(['cmdb_id','trace_name']).agg(
    window_anomaly_count=('is_anomaly','sum')
).reset_index()

earliest_anom = window_trace[window_trace['is_anomaly']].groupby(['cmdb_id','trace_name']).agg(
    earliest_anomaly_timestamp=('timestamp','min')
).reset_index()

# Merge window stats with global aggregations ensuring all groups from global are present
trace_result = global_trace_agg.merge(win_counts, on=['cmdb_id','trace_name'], how='left') \
                               .merge(win_anom, on=['cmdb_id','trace_name'], how='left') \
                               .merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for window counts with zeros and cast to int
trace_result['window_count'] = trace_result['window_count'].fillna(0).astype(int)
trace_result['window_anomaly_count'] = trace_result['window_anomaly_count'].fillna(0).astype(int)
# window_max may remain NaN; earliest_anomaly_timestamp remains NaT if none

# Select and order columns as requested
trace_result = trace_result[['cmdb_id','trace_name','count','min_timestamp','max_timestamp','p50','p95',
                             'window_count','window_anomaly_count','window_max','earliest_anomaly_timestamp']]

# 4) Sort by window_anomaly_count desc then cmdb_id and limit to top 20 rows
result_df = trace_result.sort_values(['window_anomaly_count','cmdb_id'], ascending=[False, True]).head(20)

# Display the compact result (top 20)
result_df

```
Out[5]:
```
Summary of trace analysis (thresholds computed on full series; window = 2024-01-24 14:22:25 → 14:52:25 UTC):

What I looked for
- Selected trace metrics containing "duration" or "row_count".
- Computed global P50/P95 on full series, then flagged in-window points with value >= global P95.

Top findings (highest window_anomaly_count first)
- ts-travel-service — trace.from_ts-preserve-service.row_count
  - full count 24, global p50=4, p95=7
  - window_count=24, window_anomaly_count=6, window_max=8
  - earliest anomaly: 2024-01-24T14:27:00Z
  - Interpretation: repeated row_count spikes above normal → increased request volume from preserve-service to travel-service.

- ts-route-service — trace.from_ts-basic-service.row_count
  - full count 25, p50=132, p95=144
  - window_count=25, window_anomaly_count=5, window_max=153
  - earliest anomaly: 2024-01-24T14:27:00Z
  - Interpretation: notable increases in incoming row_count to route-service (traffic spike).

- ts-train-service — trace.from_ts-basic-service.row_count
  - same profile as route-service: full count 25, p50=132, p95=144
  - window_anomaly_count=5, window_max=153, earliest 14:27:00Z
  - Interpretation: similar traffic spike pattern to train-service.

- ts-travel-service — trace.from_ts-admin-travel-service.row_count
  - full count 25, p50=10, p95=11
  - window_anomaly_count=5, window_max=13, earliest 14:38:00Z
  - Interpretation: admin-travel → travel-service saw multiple higher-than-normal batches.

- ts-auth-service — trace.from_root.row_count
  - full count 25, p50=83, p95=87
  - window_anomaly_count=4, window_max=88, earliest 14:31:00Z
  - Interpretation: increased overall row_count into auth-service.

- Several groups show 3 anomalies (examples)
  - ts-order-service trace.from_ts-security-service.row_count (p95=10 → window_max=12, earliest 14:27)
  - ts-order-service trace.from_ts-travel-service.row_count (p95=61 → window_max=67, earliest 14:29)
  - ts-travel-service trace.to_ts-order-service.row_count and trace.to_ts-seat-service.row_count (p95s exceeded; earliest 14:29)

- Duration anomalies (smaller counts, but present)
  - ts-auth-service: from_root.duration_mean (p95≈0.3991 → window_max≈0.4460; 2 anomalies; earliest 14:27)
  - ts-auth-service: duration_p95 and from_ts-auth-service.duration_* also had 2 anomalies each (very small absolute latencies but above historical P95)
  - ts-order-service: from_root.duration_mean/p95 had 2 anomalies (window_max slightly above global p95 at ~14:48)

Overall interpretation
- The dominant signal is increased row_count (traffic) between services around ~14:27–14:31 and another cluster around ~14:29–14:38. ts-travel-service, ts-route-service, ts-train-service, ts-auth-service, and ts-order-service all show elevated row_count activity relative to their historical P95s.
- Some trace durations also exceeded their historical P95s (usually 1–2 points), indicating transient latency increases that may be associated with the traffic bursts.

Recommended next steps
- Investigate the traffic source patterns around 14:27–14:31 and 14:29–14:38 (which upstream service(s) injected extra load).
- Correlate these trace row_count spikes with application metrics (CPU, memory, socket) and logs for the implicated services (ts-travel, ts-route, ts-train, ts-auth, ts-order).
- If impact is confirmed, collect detailed spans/traces for those time ranges and consider temporary scaling or throttling of the upstream callers.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                    trace_name  count             min_timestamp             max_timestamp         p50         p95  window_count  window_anomaly_count  window_max earliest_anomaly_timestamp
77  ts-travel-service      trace.from_ts-preserve-service.row_count     24 2024-01-24 14:25:00+00:00 2024-01-24 14:48:00+00:00    4.000000    7.000000            24                     6    8.000000  2024-01-24 14:27:00+00:00
38   ts-route-service         trace.from_ts-basic-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  132.000000  144.000000            25                     5  153.000000  2024-01-24 14:27:00+00:00
53   ts-train-service         trace.from_ts-basic-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  132.000000  144.000000            25                     5  153.000000  2024-01-24 14:27:00+00:00
71  ts-travel-service  trace.from_ts-admin-travel-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00   10.000000   11.000000            25                     5   13.000000  2024-01-24 14:38:00+00:00
2     ts-auth-service                     trace.from_root.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00   83.000000   87.000000            25                     4   88.000000  2024-01-24 14:31:00+00:00
26   ts-order-service      trace.from_ts-security-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    6.000000   10.000000            25                     3   12.000000  2024-01-24 14:27:00+00:00
29   ts-order-service        trace.from_ts-travel-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00   48.000000   61.000000            25                     3   67.000000  2024-01-24 14:29:00+00:00
86  ts-travel-service           trace.to_ts-order-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00   48.000000   61.000000            25                     3   67.000000  2024-01-24 14:29:00+00:00
92  ts-travel-service            trace.to_ts-seat-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00   96.000000  122.000000            25                     3  134.000000  2024-01-24 14:29:00+00:00
0     ts-auth-service                 trace.from_root.duration_mean     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    0.306244    0.399100            25                     2    0.446003  2024-01-24 14:27:00+00:00
1     ts-auth-service                  trace.from_root.duration_p95     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    0.692916    1.231307            25                     2    1.406363  2024-01-24 14:27:00+00:00
3     ts-auth-service      trace.from_ts-auth-service.duration_mean     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    0.002645    0.004167            25                     2    0.004428  2024-01-24 14:25:00+00:00
4     ts-auth-service       trace.from_ts-auth-service.duration_p95     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    0.002041    0.005158            25                     2    0.005789  2024-01-24 14:25:00+00:00
5     ts-auth-service          trace.from_ts-auth-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  334.000000  349.600000            25                     2  352.000000  2024-01-24 14:36:00+00:00
6     ts-auth-service        trace.to_ts-auth-service.duration_mean     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    0.002645    0.004167            25                     2    0.004428  2024-01-24 14:25:00+00:00
7     ts-auth-service         trace.to_ts-auth-service.duration_p95     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    0.002041    0.005158            25                     2    0.005789  2024-01-24 14:25:00+00:00
8     ts-auth-service            trace.to_ts-auth-service.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00  334.000000  349.600000            25                     2  352.000000  2024-01-24 14:36:00+00:00
9    ts-order-service                 trace.from_root.duration_mean     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    0.490972    0.594956            25                     2    0.604748  2024-01-24 14:48:00+00:00
10   ts-order-service                  trace.from_root.duration_p95     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    0.518885    0.671230            25                     2    0.689880  2024-01-24 14:48:00+00:00
11   ts-order-service                     trace.from_root.row_count     25 2024-01-24 14:25:00+00:00 2024-01-24 14:49:00+00:00    8.000000   10.800000            25                     2   12.000000  2024-01-24 14:26:00+00:00```
```